# Ghana Curriculum RAG Generation Evaluation

This pilot evaluates generated answers against their retrieved contexts with Ragas `Faithfulness` and `ResponseRelevancy`, then records per-item results in Langfuse. It uses 20 curated questions from the Basic 4–6 Maths and Science PDFs.

The fixed retrieval baseline is BM25 + Nomic dense search with reciprocal-rank fusion. PDFs are parsed by Docling and chunked with `HybridChunker(max_tokens=512, repeat_table_header=True, merge_peers=True, always_emit_headings=True)`. The Nomic model is shared between retrieval and Ragas embeddings; no ground-truth answers are included.

This notebook is a small pilot, not a retrieval benchmark. The chunking settings came from a synthetic fixture; this notebook checks real-PDF chunk output and evaluates generated answers, but does not establish that this retriever or chunker is best for the corpus. Configure the local PDF path and API credentials below before running model-backed cells.

In [1]:
from pathlib import Path
import json
import os

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ghana_curriculum_questions.json").is_file():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "02-experiments" / "maanas-ragas-eval"

QUESTION_PATH = NOTEBOOK_DIR / "ghana_curriculum_questions.json"
GHANA_DATA_DIR = Path(
    os.environ.get(
        "GHANA_ACADEMIC_DATA_DIR",
        str(Path.home() / "Documents" / "terrier-ta" / "data" / "ghana-academic-data"),
    )
).expanduser()
PDF_NAMES = (
    "MATHS-UPPER-PRIMARY-B4-B6.pdf",
    "SCIENCE-UPPER-PRIMARY-B4-B6.pdf",
)
PDF_PATHS = {name: GHANA_DATA_DIR / name for name in PDF_NAMES}
QUESTION_ITEMS = json.loads(QUESTION_PATH.read_text(encoding="utf-8"))

assert len(QUESTION_ITEMS) == 20
missing_pdfs = [str(path) for path in PDF_PATHS.values() if not path.is_file()]
if missing_pdfs:
    raise FileNotFoundError(
        "Selected Ghana PDFs were not found. Set GHANA_ACADEMIC_DATA_DIR. "
        f"Missing: {missing_pdfs}"
    )

print(f"Questions: {len(QUESTION_ITEMS)}")
print(f"PDFs: {', '.join(PDF_PATHS)}")

Questions: 20
PDFs: MATHS-UPPER-PRIMARY-B4-B6.pdf, SCIENCE-UPPER-PRIMARY-B4-B6.pdf


## 1. Environment and credentials

Install the optional evaluation stack once in this notebook's kernel:

```bash
pip install -r 02-experiments/maanas-ragas-eval/requirements.txt
```

Set `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_BASE_URL`, and an OpenAI-compatible key for generation and the Ragas judge in environment variables or a local `.env` file. Set `GHANA_ACADEMIC_DATA_DIR` if the PDF folder is not at the default path. The model is configured for CPU inference by default; its initial download and indexing will still consume time, memory, and CPU.

In [2]:
import os
import re
from collections import defaultdict
from pathlib import Path

from dotenv import load_dotenv

for env_candidate in (
    Path.cwd() / ".env",
    Path.home() / "Documents" / "terrier-ta" / ".env",
):
    if env_candidate.is_file():
        load_dotenv(env_candidate, override=False)

import numpy as np
from docling.chunking import HybridChunker
from docling.document_converter import DocumentConverter
from langchain_core.embeddings import Embeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langfuse import Evaluation, get_client
from langfuse.openai import OpenAI as LangfuseOpenAI
from ragas.dataset_schema import SingleTurnSample
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas.metrics.base import MetricWithEmbeddings, MetricWithLLM
from ragas.run_config import RunConfig
from rank_bm25 import BM25Okapi


c:\Users\saima\Documents\anacodic-agentic-cookbook\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

## 2. One Nomic embedding instance for both paths

Nomic requires `search_document:` on indexed chunks and `search_query:` on search/evaluation questions. This adapter applies those prefixes while sharing one `HuggingFaceEmbeddings` instance between retrieval and Ragas. Its default output is 768 dimensions. CPU and small batches are intentional to avoid an extra cross-encoder model.

In [ ]:
class NomicTaskEmbeddings(HuggingFaceEmbeddings):
    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        prefixed = [f"search_document: {text}" for text in texts]
        return super().embed_documents(prefixed)

    def embed_query(self, text: str) -> list[float]:
        return super().embed_query(f"search_query: {text}")


class NomicQuestionEmbeddings(Embeddings):
    """Ragas-side adapter: both original and generated questions use query mode."""

    def __init__(self, model: NomicTaskEmbeddings):
        self.model = model

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        vectors = self.model.client.encode(
            [f"search_query: {text}" for text in texts],
            normalize_embeddings=True,
            batch_size=EMBED_BATCH_SIZE,
        )
        return np.asarray(vectors, dtype=np.float32).tolist()

    def embed_query(self, text: str) -> list[float]:
        return self.model.embed_query(text)


NOMIC_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
NOMIC_DEVICE = os.environ.get("NOMIC_DEVICE", "cpu")
EMBED_BATCH_SIZE = 8
ragas_eval_embeddings = NomicTaskEmbeddings(
    model_name=NOMIC_MODEL_NAME,
    model_kwargs={"trust_remote_code": True, "device": NOMIC_DEVICE},
    encode_kwargs={"normalize_embeddings": True, "batch_size": EMBED_BATCH_SIZE},
)
ragas_embeddings = LangchainEmbeddingsWrapper(NomicQuestionEmbeddings(ragas_eval_embeddings))
EMBEDDING_DIMENSION = ragas_eval_embeddings.client.get_sentence_embedding_dimension()
EMBEDDING_MAX_TOKENS = getattr(ragas_eval_embeddings.client, "max_seq_length", None)
print(
    f"Embedding model: {NOMIC_MODEL_NAME}; dimension={EMBEDDING_DIMENSION}; "
    f"device={NOMIC_DEVICE}; model_max_seq_length={EMBEDDING_MAX_TOKENS}"
)

## 3. Extract and chunk the two source PDFs

Docling parses each PDF into a structured document. The benchmark-recommended HybridChunker settings are applied to the real curriculum PDFs; `contextualize()` output, including heading context, is the canonical text embedded and retrieved. Page and heading provenance stay attached to each chunk.

In [ ]:
MAX_CHUNK_TOKENS = 512
hybrid_chunker = HybridChunker(
    max_tokens=MAX_CHUNK_TOKENS,
    repeat_table_header=True,
    merge_peers=True,
    always_emit_headings=True,
)
pdf_converter = DocumentConverter()
CHUNKS: list[dict] = []

for source_name, pdf_path in PDF_PATHS.items():
    document = pdf_converter.convert(str(pdf_path)).document
    source_chunks = list(hybrid_chunker.chunk(document))
    for source_index, chunk in enumerate(source_chunks):
        raw_text = (getattr(chunk, "text", "") or "").strip()
        embed_text = (hybrid_chunker.contextualize(chunk) or raw_text).strip()
        if not embed_text:
            continue

        meta = getattr(chunk, "meta", None)
        headings = [str(value).strip() for value in (getattr(meta, "headings", None) or [])]
        page_numbers: set[int] = set()
        for item in getattr(meta, "doc_items", None) or []:
            for provenance in getattr(item, "prov", None) or []:
                page_number = getattr(provenance, "page_no", None)
                if page_number is not None:
                    page_numbers.add(int(page_number))

        CHUNKS.append(
            {
                "chunk_id": f"{source_name}::{source_index}",
                "source_pdf": source_name,
                "source_chunk_index": source_index,
                "pages": sorted(page_numbers),
                "headings": headings,
                "text": embed_text,
            }
        )

if not CHUNKS:
    raise RuntimeError("Docling produced no chunks from the selected PDFs.")

embedding_tokenizer = ragas_eval_embeddings.client.tokenizer
def _nomic_input_token_count(text: str) -> int:
    encoded = embedding_tokenizer(f"search_document: {text}", truncation=False)
    return len(encoded["input_ids"])

oversized = [
    (chunk["chunk_id"], _nomic_input_token_count(chunk["text"]))
    for chunk in CHUNKS
    if EMBEDDING_MAX_TOKENS and _nomic_input_token_count(chunk["text"]) > EMBEDDING_MAX_TOKENS
]
if oversized:
    raise ValueError(
        f"{len(oversized)} contextualized chunks exceed the model input limit "
        f"({EMBEDDING_MAX_TOKENS} tokens); inspect these chunks before embedding."
    )

chunk_vectors: list[list[float]] = []
for batch_start in range(0, len(CHUNKS), EMBED_BATCH_SIZE):
    batch = CHUNKS[batch_start : batch_start + EMBED_BATCH_SIZE]
    chunk_vectors.extend(ragas_eval_embeddings.embed_documents([c["text"] for c in batch]))

CHUNK_VECTORS = np.asarray(chunk_vectors, dtype=np.float32)
if CHUNK_VECTORS.shape != (len(CHUNKS), EMBEDDING_DIMENSION):
    raise ValueError(f"Unexpected embedding matrix shape: {CHUNK_VECTORS.shape}")

print(f"Indexed {len(CHUNKS)} chunks from {len(PDF_PATHS)} PDFs; vectors={CHUNK_VECTORS.shape}.")
print(f"Max contextualized Nomic input: {max(_nomic_input_token_count(c['text']) for c in CHUNKS)} tokens.")

## 4. Fixed hybrid retriever: BM25 + Nomic dense search

The selected baseline unions BM25 and Nomic-vector candidates and fuses their ranks with reciprocal-rank fusion (RRF). It deliberately does not load a separate cross-encoder reranker, keeping local compute lower. This is the fixed retriever used to produce contexts for answer evaluation, not a claim that it is optimal.

In [ ]:
def _tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.casefold())


BM25_INDEX = BM25Okapi([_tokenize(chunk["text"]) for chunk in CHUNKS])
RECIPROCAL_RANK_CONSTANT = 60
CANDIDATES_PER_RETRIEVER = 20
DEFAULT_TOP_K = 5


def retrieve_hybrid(query: str, top_k: int = DEFAULT_TOP_K) -> list[dict]:
    query_tokens = _tokenize(query)
    if not query_tokens:
        return []

    candidate_count = min(CANDIDATES_PER_RETRIEVER, len(CHUNKS))
    bm25_scores = BM25_INDEX.get_scores(query_tokens)
    bm25_order = sorted(range(len(CHUNKS)), key=lambda i: (-bm25_scores[i], i))[:candidate_count]

    query_vector = np.asarray(ragas_eval_embeddings.embed_query(query), dtype=np.float32)
    dense_scores = CHUNK_VECTORS @ query_vector
    dense_order = sorted(range(len(CHUNKS)), key=lambda i: (-dense_scores[i], i))[:candidate_count]

    fused_scores: dict[int, float] = defaultdict(float)
    for ranked_indices in (bm25_order, dense_order):
        for rank, chunk_index in enumerate(ranked_indices, start=1):
            fused_scores[chunk_index] += 1.0 / (RECIPROCAL_RANK_CONSTANT + rank)

    ranked = sorted(fused_scores, key=lambda i: (-fused_scores[i], i))[:top_k]
    return [
        {
            **CHUNKS[index],
            "score": fused_scores[index],
            "bm25_rank": bm25_order.index(index) + 1 if index in bm25_order else None,
            "dense_rank": dense_order.index(index) + 1 if index in dense_order else None,
        }
        for index in ranked
    ]


retrieval_smoke = retrieve_hybrid(QUESTION_ITEMS[0]["input"]["question"])
assert retrieval_smoke, "The fixed hybrid retriever returned no chunks for the smoke question."
print("Smoke retrieval:", [(r["source_pdf"], r["pages"], round(r["score"], 4)) for r in retrieval_smoke])

## 5. Generate answers from retrieved contexts

Each Langfuse task run performs the same fixed BM25+dense retrieval, sends only the top five contextualized chunks to the generator, and returns the answer together with those exact contexts. If retrieval returns no chunks, the prompt instructs the model to abstain.

In [ ]:
GENERATION_MODEL = os.environ.get("RAG_GENERATION_MODEL", "gpt-4.1-mini")
TOP_K = 5
generation_client = LangfuseOpenAI()


def rag_task(*, item, **kwargs):
    question = str(item.input.get("question") or "").strip()
    if not question:
        raise ValueError("Dataset item is missing input.question")

    retrieved = retrieve_hybrid(question, top_k=TOP_K)
    contexts = [chunk["text"] for chunk in retrieved]
    context_block = "\n\n---\n\n".join(contexts) if contexts else "(No context was retrieved.)"
    response = generation_client.chat.completions.create(
        model=GENERATION_MODEL,
        temperature=0,
        max_tokens=600,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer the question using only the supplied curriculum context. "
                    "If the context does not support an answer, say that the provided "
                    "curriculum context is insufficient. Do not invent curriculum details."
                ),
            },
            {
                "role": "user",
                "content": f"Context:\n{context_block}\n\nQuestion: {question}",
            },
        ],
    )
    answer = response.choices[0].message.content or ""
    return {
        "answer": answer,
        "contexts": contexts,
        "retrieved_chunks": [
            {
                "chunk_id": chunk["chunk_id"],
                "source_pdf": chunk["source_pdf"],
                "pages": chunk["pages"],
                "headings": chunk["headings"],
            }
            for chunk in retrieved
        ],
    }


print(f"Generator configured: {GENERATION_MODEL}; top_k={TOP_K}")

## 6. Ragas metrics: Faithfulness and ResponseRelevancy

Faithfulness checks whether answer claims are supported by the retrieved contexts. ResponseRelevancy checks whether the answer addresses the question using the configured embeddings and judge model. Neither metric receives a reference answer. The judge is configured to one worker to keep the pilot's model-call concurrency modest.

In [ ]:
JUDGE_MODEL = os.environ.get("RAGAS_JUDGE_MODEL", "gpt-4.1-mini")
ragas_judge_llm = LangchainLLMWrapper(ChatOpenAI(model=JUDGE_MODEL, temperature=0))
ragas_metrics = [Faithfulness(), ResponseRelevancy()]

for metric in ragas_metrics:
    if isinstance(metric, MetricWithLLM):
        metric.llm = ragas_judge_llm
    if isinstance(metric, MetricWithEmbeddings):
        metric.embeddings = ragas_embeddings
    metric.init(RunConfig(max_workers=1))


def make_ragas_evaluator(metric):
    async def evaluator(*, input, output, **kwargs):
        sample = SingleTurnSample(
            user_input=input["question"],
            retrieved_contexts=output["contexts"],
            response=output["answer"],
        )
        score = await metric.single_turn_ascore(sample)
        return Evaluation(name=metric.name, value=float(score))

    return evaluator


ragas_evaluators = [make_ragas_evaluator(metric) for metric in ragas_metrics]
print(f"Evaluators ready: {[metric.name for metric in ragas_metrics]}")

## 7. Create the Langfuse smoke dataset and run one item

The dataset input contains only the question; source provenance is metadata and there are no `expected_output` or reference-answer fields. This cell makes one task run and two metric evaluations so you can verify the complete path before spending tokens on all 20 questions. Run it once; if the dataset name already exists, choose a new versioned name.

In [ ]:
LANGFUSE_DATASET_SMOKE = "ghana-curriculum-generation-smoke-v1"
missing_credentials = [
    name
    for name in ("LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL", "OPENAI_API_KEY")
    if not os.environ.get(name)
]
if missing_credentials:
    raise RuntimeError(f"Set these environment variables before running: {missing_credentials}")

langfuse = get_client()
langfuse.create_dataset(name=LANGFUSE_DATASET_SMOKE)
for question_item in QUESTION_ITEMS[:1]:
    langfuse.create_dataset_item(
        dataset_name=LANGFUSE_DATASET_SMOKE,
        id=f"{LANGFUSE_DATASET_SMOKE}:{question_item['id']}",
        input=question_item["input"],
        metadata=question_item["metadata"],
    )

smoke_dataset = langfuse.get_dataset(LANGFUSE_DATASET_SMOKE)
smoke_result = smoke_dataset.run_experiment(
    name="Ghana curriculum generation smoke",
    description="One source-grounded question; BM25 + Nomic dense RRF; Faithfulness and ResponseRelevancy.",
    task=rag_task,
    evaluators=ragas_evaluators,
    max_concurrency=1,
)
print(smoke_result.format())

## 8. Publish and run all 20 questions

After the smoke run succeeds and its traces/context look correct, create the versioned 20-item dataset and run the full pilot. The runner is limited to one concurrent task. Langfuse's Hobby quota counts traces, observations, and scores; the generator and Ragas judge also incur separate model-provider usage.

In [ ]:
LANGFUSE_DATASET_PILOT = "ghana-curriculum-generation-pilot-v1"
langfuse.create_dataset(name=LANGFUSE_DATASET_PILOT)
for question_item in QUESTION_ITEMS:
    langfuse.create_dataset_item(
        dataset_name=LANGFUSE_DATASET_PILOT,
        id=f"{LANGFUSE_DATASET_PILOT}:{question_item['id']}",
        input=question_item["input"],
        metadata=question_item["metadata"],
    )

pilot_dataset = langfuse.get_dataset(LANGFUSE_DATASET_PILOT)
pilot_result = pilot_dataset.run_experiment(
    name="Ghana curriculum generation pilot",
    description="20 questions; Docling HybridChunker 512; BM25 + Nomic dense RRF; Faithfulness and ResponseRelevancy.",
    task=rag_task,
    evaluators=ragas_evaluators,
    max_concurrency=1,
)
print(pilot_result.format())